# VascuMap Batch Pipeline

Runs the full VascuMap pipeline on every image in a source folder.  
- **LIF files**: iterates through each image within the file  
- **TIF/TIFF files**: one pipeline run per file  
- Per-image output folder under `Z:\Bel\Vascumap_Outputs\<image_name>\`

**Default outputs** (always saved):
| File | Description |
|------|-------------|
| `*_overlay_geometry_0.tif` | 2-D device overlay with inner/outer geometry |
| `*_skeleton_overview.png` | Skeleton overview plot |
| `*_analysis_metrics.csv` | Quantitative vascular network metrics |

**Extra outputs** (when `save_all_interim = True`):
| File | Description |
|------|-------------|
| `*_cropped_stack_aligned.npy` | Brightfield stack at 2 µm iso |
| `*_vessel_translation_aligned.npy` | Pix2Pix image translation |
| `*_clean_segmentation.npy` | Cleaned binary vessel segmentation |
| `*_skeleton.npy` | Skeleton from pruned graph |
| `*_holes.npy` | Binary pore mask |
| `*_hole_labels_per_slice.npy` | Per-slice labelled pores |
| `*_hole_distance_per_slice_um.npy` | Inscribed-radius distance map |
| `*_full_graph_skeleton.npy` | Pre-pruning skeleton |
| `*_vessel_mask.npy` | Raw vessel mask |
| `*_graph_nodes.npz` | Sprout/junction node coords |
| `*_clean_graph.pkl` | NetworkX graph (pickle) |

In [1]:
# ── Configuration ──────────────────────────────────────────────────────────────
# Point source_dir at the folder containing your .lif / .tif / .tiff files.
# All outputs land under output_base in per-image sub-folders.

source_dir  = r"Z:\Bel\Vascumap_Example_Lifs\training_data\original_images\test_in\second_run"    
output_base = r"Z:\Bel\Vascumap_Outputs"

# Set True for images whose filename contains "marina" (enables organoid masking).
# If None, the notebook auto-detects from the filename.
force_mask_central_region = None   # True / False / None (auto)

# Device width in micrometres (passed to device segmentation)
device_width_um = 35.0

# Channel index to use for multi-channel images (0-based, default 0)
channel = 0

# Set True to save extra volumes for full napari visualisation
# (holes, pore labels, full-graph skeleton, graph node coords, etc.)
save_all_interim = True

In [2]:
# ── Imports ────────────────────────────────────────────────────────────────────
import sys
from pathlib import Path

# Add bel_vascumap to the path so we can import the pipeline
sys.path.insert(0, str(Path(r"C:\Users\taylorhearn\git_repos\vascumap\bel_vascumap")))

import numpy as np
import tifffile
from liffile import LifFile

from vascumap import VascuMap
from utils import resize_dask

W0317 12:49:05.153000 619616 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
c:\Users\taylorhearn\AppData\Local\miniconda3\envs\clean_vascumap\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.0)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
c:\Users\taylorhearn\AppData\Local\miniconda3\envs\clean_vascumap\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# ── Load models once (shared across all batch jobs) ────────────────────────────
from models import Pix2Pix, load_segmentation_model

shared_model_p2p = Pix2Pix(
    model_path=r"C:\Users\taylorhearn\git_repos\vascumap\luca_models\epoch=117-val_g_psnr=20.47-val_g_ssim=0.62.ckpt"
)
shared_model_unet = load_segmentation_model(
    r"C:\Users\taylorhearn\git_repos\vascumap\luca_models\best_full.pth"
)
print("Models loaded.")

In [3]:
# ── Discover image files and build job list ────────────────────────────────────
source = Path(source_dir)
if not source.is_dir():
    raise FileNotFoundError(f"source_dir does not exist: {source}")

image_files = sorted(
    p for p in source.iterdir()
    if p.is_file() and p.suffix.lower() in (".lif", ".tif", ".tiff")
)

# Each job is (path, image_index, should_mask)
jobs = []
for p in image_files:
    should_mask = (
        force_mask_central_region
        if force_mask_central_region is not None
        else ("marina" in p.name.lower())
    )
    if p.suffix.lower() == ".lif":
        try:
            with LifFile(p) as lif:
                n_images = len(lif.images)
            for idx in range(n_images):
                jobs.append((p, idx, should_mask))
        except Exception as exc:
            print(f"[SKIP] Could not inspect {p.name}: {exc}")
    else:
        jobs.append((p, 0, should_mask))

print(f"Source: {source}")
print(f"Files found: {len(image_files)}  |  Total jobs: {len(jobs)}")
for i, (p, idx, mask) in enumerate(jobs):
    tag = f" (LIF image {idx})" if p.suffix.lower() == ".lif" else ""
    print(f"  {i+1}. {p.name}{tag}  mask_central_region={mask}")

Source: Z:\Bel\Vascumap_Example_Lifs\training_data\original_images\test_in\second_run
Files found: 4  |  Total jobs: 4
  1. farid_new_flow_double_channel.tif  mask_central_region=False
  2. farid_new_flow_double_channel_2.tif  mask_central_region=False
  3. farid_new_flow_single_channel.tif  mask_central_region=False
  4. farid_new_flow_single_channel_2.tif  mask_central_region=False


In [ ]:
# ── Processing function ────────────────────────────────────────────────────────

def process_and_save(image_path: Path, image_index: int, output_base: str,
                     mask_central_region: bool = False,
                     save_all_interim: bool = False,
                     channel: int = 0,
                     model_p2p=None,
                     model_unet=None):
    """Run full VascuMap pipeline and save aligned outputs to a per-image folder."""
    vm = VascuMap(
        use_device_segmentation_app=False,
        image_source_path=str(image_path),
        image_index=image_index,
        device_width_um=device_width_um,
        mask_central_region=mask_central_region,
        channel=channel,
        model_p2p=model_p2p,
        model_unet=model_unet,
    )

    # Build a short human-readable name
    if image_path.suffix.lower() == ".lif":
        vm.image_name = f"{image_path.stem}_img{image_index}_{vm.image_name or 'image'}"
    else:
        vm.image_name = f"{image_path.stem}_{vm.image_name or 'image'}"

    output_dir = Path(output_base) / vm.image_name
    print(f"  Output → {output_dir}")

    vm.pipeline(output_dir=output_dir, save_all_interim=save_all_interim)
    return vm

In [5]:
# ── Run batch ──────────────────────────────────────────────────────────────────
MAX_RETRIES = 2
results = []          # (short_name, status, message)

for i, (image_path, image_index, mask_flag) in enumerate(jobs, start=1):
    tag = f" (LIF #{image_index})" if image_path.suffix.lower() == ".lif" else ""
    print(f"\n{'='*70}")
    print(f"[{i}/{len(jobs)}] {image_path.name}{tag}  mask_central_region={mask_flag}")
    print(f"{'='*70}")

    last_exc = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            vm = process_and_save(
                image_path=image_path,
                image_index=image_index,
                output_base=output_base,
                mask_central_region=mask_flag,
                save_all_interim=save_all_interim,
                channel=channel,
                model_p2p=shared_model_p2p,
                model_unet=shared_model_unet,
            )
            results.append((vm.image_name or image_path.stem, "OK", ""))
            last_exc = None
            break
        except Exception as exc:
            last_exc = exc
            if attempt < MAX_RETRIES:
                print(f"  ⚠ Attempt {attempt} failed: {exc}  — retrying...")
            else:
                print(f"  ✗ FAILED after {MAX_RETRIES} attempts: {exc}")

    if last_exc is not None:
        results.append((image_path.name + tag, "FAILED", str(last_exc)))

# ── Summary ───────────────────────────────────────────────────────────────
print(f"\n{'='*70}")
n_ok = sum(1 for _, s, _ in results if s == "OK")
print(f"Batch complete: {n_ok}/{len(results)} succeeded")
if any(s == "FAILED" for _, s, _ in results):
    print("\nFailed images:")
    for name, status, msg in results:
        if status == "FAILED":
            print(f"  - {name}: {msg}")


[1/4] farid_new_flow_double_channel.tif  mask_central_region=False
  Output → Z:\Bel\Vascumap_Outputs\farid_new_flow_double_channel_farid_new_flow_double_channel
Initial z votes {0: 0, 1: 0, 2: 1, 3: 0, 4: 0, 5: 1, 6: 0, 7: 0, 8: 3, 9: 4, 10: 2, 11: 6, 12: 7, 13: 3, 14: 10, 15: 6, 16: 13, 17: 4, 18: 2, 19: 15, 20: 10, 21: 7, 22: 2, 23: 4, 24: 0, 25: 0, 26: 0, 27: 0, 28: 0, 29: 0, 30: 0, 31: 0, 32: 0, 33: 0}
First cropping to z: {7: 0, 8: 3, 9: 4, 10: 2, 11: 6, 12: 7, 13: 3, 14: 10, 15: 6, 16: 13, 17: 4, 18: 2, 19: 15, 20: 10, 21: 7, 22: 2, 23: 4}
Stack width 169.99216969696968


 88%|████████▊ | 1367/1549 [03:21<00:25,  7.08it/s]Exception in callback BaseAsyncIOLoop._handle_events(1336, 1)
handle: <Handle BaseAsyncIOLoop._handle_events(1336, 1)>
Traceback (most recent call last):
  File "c:\Users\taylorhearn\AppData\Local\miniconda3\envs\clean_vascumap\Lib\asyncio\events.py", line 84, in _run
    self._context.run(self._callback, *self._args)
  File "c:\Users\taylorhearn\AppData\Local\miniconda3\envs\clean_vascumap\Lib\site-packages\tornado\platform\asyncio.py", line 208, in _handle_events
    handler_func(fileobj, events)
  File "c:\Users\taylorhearn\AppData\Local\miniconda3\envs\clean_vascumap\Lib\site-packages\zmq\eventloop\zmqstream.py", line 600, in _handle_events
    self._handle_recv()
  File "c:\Users\taylorhearn\AppData\Local\miniconda3\envs\clean_vascumap\Lib\site-packages\zmq\eventloop\zmqstream.py", line 629, in _handle_recv
    self._run_callback(callback, msg)
  File "c:\Users\taylorhearn\AppData\Local\miniconda3\envs\clean_vascumap\Lib\site-pack

strong contiguous vote planes 11-17
Running skeletonisation and analysis...


Processing chunks: 100%|██████████| 1/1 [00:02<00:00,  2.92s/it]


 chip_volume_um3  vessel_volume_um3  vessel_volume_fraction  total_vessel_length_um  vessel_length_per_chip_volume_um_inverse2  sprouts_per_vessel_length_um_inverse  junctions_per_vessel_length_um_inverse  skeleton_fractal_dimension  skeleton_lacunarity  median_sprout_and_branch_tortuosity  p90_minus_p10_sprout_and_branch_tortuosity  median_sprout_and_branch_median_cs_area_um2  p90_minus_p10_sprout_and_branch_median_cs_area_um2  median_junction_dist_nearest_junction_um  p90_minus_p10_junction_dist_nearest_junction_um  median_sprout_dist_nearest_endpoint_um  p90_minus_p10_sprout_dist_nearest_endpoint_um  total_internal_pore_count  internal_pore_area_fraction_in_filled_vascular_area  median_internal_pore_area_um2  p90_minus_p10_internal_pore_area_um2  median_internal_pore_max_inscribed_radius_um  p90_minus_p10_internal_pore_max_inscribed_radius_um  sprouts_per_chip_volume_um_inverse3  junctions_per_chip_volume_um_inverse3  total_internal_pore_density_per_vessel_volume_um_inverse3
    174

...

  Saved all interim outputs for napari visualisation
  ✓ Done: farid_new_flow_single_channel_farid_new_flow_single_channel

[4/4] farid_new_flow_single_channel_2.tif  mask_central_region=False
  Output → Z:\Bel\Vascumap_Outputs\farid_new_flow_single_channel_2_farid_new_flow_single_channel_2
Initial z votes {0: 0, 1: 0, 2: 1, 3: 0, 4: 1, 5: 1, 6: 1, 7: 1, 8: 2, 9: 5, 10: 1, 11: 4, 12: 2, 13: 2, 14: 3, 15: 6, 16: 7, 17: 7, 18: 6, 19: 9, 20: 5, 21: 11, 22: 11, 23: 8, 24: 3, 25: 1, 26: 1, 27: 1, 28: 0, 29: 0, 30: 0, 31: 0}
First cropping to z: {4: 1, 5: 1, 6: 1, 7: 1, 8: 2, 9: 5, 10: 1, 11: 4, 12: 2, 13: 2, 14: 3, 15: 6, 16: 7, 17: 7, 18: 6, 19: 9, 20: 5, 21: 11, 22: 11, 23: 8, 24: 3, 25: 1, 26: 1, 27: 1}
Stack width 239.98892903225806


Processing chunks: 100%|██████████| 4/4 [00:35<00:00,  9.00s/it]


strong contiguous vote planes 14-24
Running skeletonisation and analysis...


Processing chunks: 100%|██████████| 1/1 [00:02<00:00,  2.97s/it]


 chip_volume_um3  vessel_volume_um3  vessel_volume_fraction  total_vessel_length_um  vessel_length_per_chip_volume_um_inverse2  sprouts_per_vessel_length_um_inverse  junctions_per_vessel_length_um_inverse  skeleton_fractal_dimension  skeleton_lacunarity  median_sprout_and_branch_tortuosity  p90_minus_p10_sprout_and_branch_tortuosity  median_sprout_and_branch_median_cs_area_um2  p90_minus_p10_sprout_and_branch_median_cs_area_um2  median_junction_dist_nearest_junction_um  p90_minus_p10_junction_dist_nearest_junction_um  median_sprout_dist_nearest_endpoint_um  p90_minus_p10_sprout_dist_nearest_endpoint_um  total_internal_pore_count  internal_pore_area_fraction_in_filled_vascular_area  median_internal_pore_area_um2  p90_minus_p10_internal_pore_area_um2  median_internal_pore_max_inscribed_radius_um  p90_minus_p10_internal_pore_max_inscribed_radius_um  sprouts_per_chip_volume_um_inverse3  junctions_per_chip_volume_um_inverse3  total_internal_pore_density_per_vessel_volume_um_inverse3
    273